<a href="https://colab.research.google.com/github/shahzad-jatoi/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahzad-jatoi/flyrank-ml-internship-starter/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [7]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shahzad-jatoi/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    os.chdir("/content")
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(f"/content/{REPO_DIR}")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "scikit-learn"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import duckdb, pandas as pd, numpy as np, json, os

if IN_COLAB:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
DIM_CONTENT = f"{WAREHOUSE}/dim_content.parquet"
print("Setup done.")

Setup done.


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding 1: "The Freshness Multiplier" 365+ day content refreshed within 30 days shows a 3.2x health boost (10.7 → 34.5) and 57x more impressions (71 → 4,039).**

Methodology question: where does the "refreshed" label come from, and is there a selection effect in which pages get refreshed? The paper itself flags that the 361+ freshness bucket has only 1 declining page against 283 growing ones an extremely thin sample. A likely alternative explanation: pages chosen for refresh by an editorial team are probably not a random sample of old content they're likely to be pages that still have residual backlinks, brand authority, or historical traffic that made them worth the editorial time in the first place. If so, some of the 57x lift could reflect "we picked pages likely to recover" rather than "refreshing caused the recovery." A stronger design would compare refreshed pages against a matched set of similarly old, similarly authoritative pages that were *not* refreshed in the same window, rather than comparing refreshed pages only against their own past performance.

**Finding 2: ML Appendix Random Forest feature importance for predicting Health Score, with Average Position at 43% and Impressions at 32%.**

Methodology question: the paper is admirably upfront that "the target itself is partly constructed from some of these inputs" and looking at the Health Score formula (impressions 30pts + position 30pts + CTR 20pts + scroll depth 20pts), Average Position and Impressions are literally two of the four components being summed into Health Score. So a Random Forest finding that these two features are the top predictors of Health Score is close to circular: the model is partly just reconstructing a known formula, not discovering a novel driver of quality. The paper's own caveat ("read this as model behavior, not as a standalone optimization order") is the right instinct the real methodology question is why include formula components as model *features* at all, rather than only feeding in inputs that are genuinely independent of the target (e.g. word count, freshness, AI sessions) if the goal is to learn what actually drives health, not to re-derive the formula.

(Constructive framing: both of these are the same category of question I'd want asked of my own Week 5 model "does the validation design rule out the boring alternative explanation" and "is any input secretly encoding the label" not "this paper is wrong.")

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.




## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

def load_features(month_str, ref_date):
    fact_path = f"{WAREHOUSE}/fact_content_daily_performance/month={month_str}/data_0.parquet"
    return con.sql(f"""
        WITH agg AS (
            SELECT content_hash_id,
                   AVG(gsc_impressions) as avg_impressions,
                   AVG(gsc_clicks) as avg_clicks,
                   AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions,0)) as avg_ctr,
                   AVG(gsc_avg_position) as avg_position
            FROM read_parquet('{fact_path}')
            WHERE gsc_data_available IS TRUE
            GROUP BY content_hash_id
        )
        SELECT a.*, c.word_count, c.search_volume,
               DATE_DIFF('day', c.content_updated_date, DATE '{ref_date}') as days_since_update
        FROM agg a
        JOIN read_parquet('{DIM_CONTENT}') c ON a.content_hash_id = c.content_hash_id
        WHERE c.is_deleted IS NOT TRUE AND a.avg_impressions >= 10
    """).df()

def load_outcome(month_str):
    fact_path = f"{WAREHOUSE}/fact_content_daily_performance/month={month_str}/data_0.parquet"
    return con.sql(f"""
        SELECT content_hash_id, AVG(gsc_avg_position) as future_avg_position
        FROM read_parquet('{fact_path}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    """).df()

FEATURES = ["avg_impressions", "avg_clicks", "avg_ctr", "avg_position", "word_count", "search_volume", "days_since_update"]

feb = load_features("2026-02", "2026-02-28").merge(load_outcome("2026-03"), on="content_hash_id")
feb["declined"] = (feb["future_avg_position"] > feb["avg_position"]).astype(int)
apr = load_features("2026-04", "2026-04-30").merge(load_outcome("2026-05"), on="content_hash_id")
apr["declined"] = (apr["future_avg_position"] > apr["avg_position"]).astype(int)

imputer = SimpleImputer(strategy="median")

# BEFORE: naive random split, pooling Feb+Apr together (ignores time structure)
pooled = pd.concat([feb, apr], ignore_index=True)
Xp = imputer.fit_transform(pooled[FEATURES])
yp = pooled["declined"].values
Xp_train, Xp_test, yp_train, yp_test = train_test_split(Xp, yp, test_size=0.3, random_state=42, stratify=yp)
naive_model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xp_train, yp_train)
naive_acc = (naive_model.predict(Xp_test) == yp_test).mean()

# AFTER: honest time-aware split (train on Feb->Mar outcome, test on fully separate Apr->May outcome)
Xtr = imputer.fit_transform(feb[FEATURES])
ytr = feb["declined"].values
Xte = imputer.transform(apr[FEATURES])
yte = apr["declined"].values
honest_model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xtr, ytr)
honest_acc = (honest_model.predict(Xte) == yte).mean()

print(f"BEFORE (naive random split, time-mixed):  accuracy = {naive_acc:.3f}")
print(f"AFTER  (honest time-aware split):          accuracy = {honest_acc:.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

BEFORE (naive random split, time-mixed):  accuracy = 0.666
AFTER  (honest time-aware split):          accuracy = 0.274


Before/after: the naive random split showed 0.669 accuracy, while the honest time-aware split showed only 0.274 a large drop, and notably below what a simple majority-class guess would achieve in the test period (see check above). This is a stronger and more concerning result than a modest accuracy drop would have been. The random split's higher number was almost certainly inflated by information leaking across nearby time periods pages in Feb and Apr are the same underlying pages with highly similar recent behavior, so a random split let the model implicitly "see" near-duplicate rows across train and test.

The honest split's low score suggests the model is not just weaker out-of-time, it may not be capturing a signal that holds up between Feb->Mar and Apr->May at all. A likely explanation is a shift in the underlying decline rate or feature distributions between the two windows (see decline-rate comparison above), which a single time-aware split can't distinguish from genuine model failure. This is the central lesson of this assignment: the naive number was actively misleading, and the honest number even though it's an uncomfortable result is the one worth reporting and investigating further, not the flattering one.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("Features used:", FEATURES)
print("\nLabel: declined = future_avg_position > avg_position")
print("future_avg_position comes from the month AFTER the feature window — confirmed separate.")

leak_check = apr[FEATURES + ["future_avg_position"]].corr()["future_avg_position"].drop("future_avg_position")
print("\nCorrelation of each feature with future_avg_position (should be moderate, not suspiciously ~1.0):")
print(leak_check.sort_values(key=abs, ascending=False))

print(f"\nFlag from Week 5: avg_position correlation with future_avg_position = {leak_check['avg_position']:.3f}")
print("Expected (position is autocorrelated month to month), but worth treating as a near-leakage risk")
print("since the label is partly defined relative to this same feature.")


Features used: ['avg_impressions', 'avg_clicks', 'avg_ctr', 'avg_position', 'word_count', 'search_volume', 'days_since_update']

Label: declined = future_avg_position > avg_position
future_avg_position comes from the month AFTER the feature window — confirmed separate.

Correlation of each feature with future_avg_position (should be moderate, not suspiciously ~1.0):
avg_position         0.818613
word_count           0.214911
avg_ctr             -0.208677
days_since_update    0.145304
avg_clicks          -0.140455
avg_impressions     -0.129086
search_volume        0.062788
Name: future_avg_position, dtype: float64

Flag from Week 5: avg_position correlation with future_avg_position = 0.819
Expected (position is autocorrelated month to month), but worth treating as a near-leakage risk
since the label is partly defined relative to this same feature.


Leakage check: avg_position shows a correlation of 0.819 with future_avg_position by far the strongest of the seven features, and high enough to be a genuine near-leakage risk rather than just background autocorrelation. No feature comes from a period after the feature window: avg_impressions, avg_clicks, avg_ctr, avg_position, word_count, search_volume, and days_since_update are all computed from the same month as the rest of the feature set, and future_avg_position is confirmed to come strictly from the following month.

Still, a 0.819 correlation between avg_position and the value the label is directly derived from (declined = future_avg_position > avg_position) means the model likely has an easy, almost mechanical shortcut available: pages already ranking poorly have less room left to decline further, so avg_position alone can predict a large share of the label without learning anything about *why* a page is likely to slip. This lines up with the Section 2 result if that near-mechanical relationship between avg_position and future_avg_position shifted even slightly between the Feb-to-Mar and Apr-to-May windows, a model trained on one window's version of that relationship could perform badly on the other, which is a plausible partial explanation for the sharp drop from 0.669 (naive split) to 0.274 (honest split). The remaining features (word_count 0.21, avg_ctr -0.21, days_since_update 0.15) are much weaker and don't show any suspiciously perfect correlation, so the leakage concern is narrowly about avg_position, not the whole feature set.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (too bold):** "The model beat the baseline rule at Precision@20, correctly flagging 19 of the top 20 declining pages, and generalizes to future data."

**Rewritten (safe language):** "In earlier testing, the model's top-20 ranked pages showed a high measured decline rate on one held-out window. However, under a stricter honest time-aware validation (training on Feb→Mar outcomes, testing on the fully separate Apr→May window), accuracy dropped sharply from 0.669 to 0.274 below what a simple majority-class guess would achieve. Combined with a 0.819 correlation between the model's strongest feature (avg_position) and the value the label is derived from, this result should not be reported as a working predictive model. At most, it is an observed pattern on specific historical windows that has not yet been shown to generalize, and the honest evaluation raises real doubt about whether it generalizes at all. Further investigation checking for distribution shift between windows and testing with avg_position excluded is needed before any decision-support claim can be made."

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.